In [0]:
%sql
--Average Days in Shop:
--Average days a vehicle spends in the shop, by store and service type.
SELECT
  store_id,
  service_type,
  ROUND(AVG(DATEDIFF(vehicle_out_date, vehicle_in_date)), 2) AS avgerage_days_in_shop
FROM
  workspace.gold.data_cube
WHERE
  order_status = "COMPLETED"
GROUP by
  store_id,
  service_type
ORDER BY
  store_id,
  service_type;

In [0]:
%sql
--Survey Coverage:
--For each store, number of surveys sent versus number of surveys responded.
--select * from workspace.gold.data_cube;
select
  store_id,
  sum(
    case
      when (responded_flag is not null) then 1
      else 0
    end
  ) as surveys_sent,
  sum(
    case
      when responded_flag is true then 1
      else 0
    end
  ) as surveys_responded,
  round((surveys_responded * 100.0) / surveys_sent, 2) as surveys_coverage_percentage
from
  workspace.gold.data_cube
group by
  store_id;

In [0]:
%sql
--Revenue vs Budget:
--Monthly revenue compared against budget by manager, and ranking of managers by budget achievement.
WITH revenue_and_budget_data AS (
  SELECT
    store_id,
    manager_id,
    manager_name,
    month(month) AS budget_month,
    SUM(invoice_amount) AS revenue,
    SUM(budget_amount) AS budget
  FROM
    workspace.gold.data_cube
  GROUP BY
    store_id,
    manager_id,
    manager_name,
    month(month)
)
SELECT
  *,
  budget - revenue AS variance,
  RANK() OVER (ORDER BY budget - revenue ASC) AS manager_rank
FROM
  revenue_and_budget_data;

In [0]:
%sql
--MTD Performance vs Previous MTD:
--Month-to-date revenue and number of completed orders compared with previous month-to-date, by store and manager.
WITH base AS (
  SELECT
    *,
    DATE_TRUNC('month', vehicle_in_date) AS month_start
  FROM
    workspace.gold.data_cube
  WHERE
    order_status = 'COMPLETED'
),
current_mtd AS (
  SELECT
    store_id,
    manager_id,
    manager_name,
    COUNT(order_id) AS orders,
    SUM(estimate_amount) AS revenue
  FROM
    base
  WHERE
    vehicle_in_date >= DATE_TRUNC('month', CURRENT_DATE)
    AND vehicle_in_date <= CURRENT_DATE
  GROUP BY
    store_id,
    manager_id,
    manager_name
),
previous_mtd AS (
  SELECT
    store_id,
    manager_id,
    manager_name,
    COUNT(order_id) AS orders,
    SUM(estimate_amount) AS revenue
  FROM
    base
  WHERE
    vehicle_in_date >= DATE_TRUNC('month', ADD_MONTHS(CURRENT_DATE, -1))
    AND vehicle_in_date < DATE_TRUNC('month', CURRENT_DATE)
  GROUP BY
    store_id,
    manager_id,
    manager_name
)
SELECT
  c.store_id,
  c.manager_id,
  c.manager_name,
  c.orders AS current_orders,
  p.orders AS previous_orders,
  c.revenue AS current_revenue,
  p.revenue AS previous_revenue
FROM
  current_mtd c
    LEFT JOIN previous_mtd p
      ON c.store_id = p.store_id
      AND c.manager_id = p.manager_id
      AND c.manager_name = p.manager_name;

In [0]:
%sql
--Top Technicians by Completion Time Accuracy:
--For each store, top 5 technicians based on accuracy of meeting promised completion dates.
WITH tech_perf AS (
  SELECT
    store_id,
    technician_id,
    COUNT(order_id) AS total_jobs,
    SUM(
      CASE
        WHEN actual_completion_date <= promised_delivery_date THEN 1
        ELSE 0
      END
    ) AS on_time_jobs
  FROM
    workspace.gold.data_cube
  GROUP BY
    store_id,
    technician_id
),
ranked AS (
  SELECT
    *,
    round((on_time_jobs * 100.0 / total_jobs), 2) AS accuracy,
    ROW_NUMBER() OVER (PARTITION BY store_id ORDER BY (on_time_jobs * 1.0 / total_jobs) DESC) AS rn
  FROM
    tech_perf
)
SELECT
  *
FROM
  ranked
WHERE
  rn <= 5;

In [0]:
%sql
-- Estimator Accuracy
--Accuracy of estimates (initial vs final) by estimator and ranking of estimators by highest accuracy.
WITH est_calc AS (
  SELECT
    invoice_id,
    estimator_id,
    estimator_name,
    MAX(
      CASE
        WHEN estimate_version_no = 1 THEN estimate_amount
      END
    ) AS initial_estimate,
    MAX(estimate_amount) AS final_estimate
  FROM
    workspace.gold.data_cube
  --WHERE estimator_id="ESTIM002"
  GROUP BY
    invoice_id,
    estimator_id,
    estimator_name
),
est_accuracy AS (
  SELECT
    estimator_id,
    estimator_name,
    AVG(
      CASE
        WHEN initial_estimate > 0 THEN (1 - final_estimate - initial_estimate) / initial_estimate
      END
    )
      * 100 AS avg_accuracy_pct
  FROM
    est_calc
  WHERE
    initial_estimate IS NOT NULL
    AND final_estimate IS NOT NULL
  GROUP BY
    estimator_id,
    estimator_name
)
SELECT
  estimator_id,
  estimator_name,
  ROUND(avg_accuracy_pct, 2) AS accuracy_pct,
  RANK() OVER (ORDER BY avg_accuracy_pct DESC) AS estimator_rank
FROM
  est_accuracy;

In [0]:
%sql
--Technician Workload
--Number of orders handled and total days in shop per technician per month and ranking of technicians by workload.
WITH workload AS (
  SELECT
    technician_id,
    technician_name,
    --store_id,
    DATE_TRUNC('month', vehicle_in_date) AS month,
    COUNT(order_id) AS total_orders,
    SUM(DATEDIFF(vehicle_out_date, vehicle_in_date)) AS total_days
  FROM
    workspace.gold.data_cube
  GROUP BY
    technician_id,
    technician_name,
    --store_id,
    DATE_TRUNC('month', vehicle_in_date)
)
SELECT
  *,
  RANK() OVER (PARTITION BY month ORDER BY total_orders DESC) AS workload_rank
FROM
  workload;

In [0]:
%sql
--Stage-wise Day Cycle Time:
--Average days spent in each stage (vehicle-in to work-start, work-start to completion, completion to delivery), by store and service type.
SELECT
  store_id,
  service_type,
  AVG(DATEDIFF(actual_work_start_date, vehicle_in_date)) as average_in_to_start_days,
  AVG(DATEDIFF(actual_completion_date, actual_work_start_date)) as average_start_to_completion_days,
  AVG(DATEDIFF(actual_delivery_date, actual_completion_date)) as average_completion_to_delivery_days
FROM
  workspace.gold.data_cube
GROUP BY
  store_id,
  service_type;

In [0]:
%sql
--Year-to-Date Revenue Growth:
--Year-to-date revenue compared with previous year YTD, and ranking of stores by YTD growth.
WITH max_date_cte AS (
  SELECT
    MAX(vehicle_in_date) AS max_date
  FROM
    workspace.gold.data_cube
),
ytd AS (
  SELECT
    store_id,
    SUM(invoice_amount) AS current_ytd_revenue
  FROM
    workspace.gold.data_cube CROSS JOIN max_date_cte
  WHERE
    vehicle_in_date >= DATE_TRUNC('year', max_date_cte.max_date)
    AND vehicle_in_date <= max_date_cte.max_date
  GROUP BY
    store_id
),
prev_ytd AS (
  SELECT
    store_id,
    SUM(invoice_amount) AS prev_ytd_revenue
  FROM
    workspace.gold.data_cube CROSS JOIN max_date_cte
  WHERE
    vehicle_in_date >= DATE_TRUNC('year', max_date_cte.max_date - INTERVAL 1 YEAR)
    AND vehicle_in_date <= DATE_TRUNC('year', max_date_cte.max_date) - INTERVAL 1 DAY
  GROUP BY
    store_id
)
SELECT
  y.store_id,
  y.current_ytd_revenue,
  p.prev_ytd_revenue,
  (y.current_ytd_revenue - p.prev_ytd_revenue) AS ytd_growth,
  ROUND(
    (y.current_ytd_revenue - p.prev_ytd_revenue) * 100.0 / NULLIF(p.prev_ytd_revenue, 0),
    2
  ) AS ytd_growth_pct,
  RANK() OVER (
      ORDER BY
        ROUND(
          (y.current_ytd_revenue - p.prev_ytd_revenue) * 100.0 / NULLIF(p.prev_ytd_revenue, 0),
          2
        ) DESC
    ) AS ytd_growth_rank
FROM
  ytd y
    LEFT JOIN prev_ytd p
      ON y.store_id = p.store_id;

In [0]:
%sql
--Survey Scores Summary:
--Overall customer survey metrics by store and ranking of stores by overall satisfaction
SELECT
  store_id,
  ROUND(AVG(overall_satisfaction_rating), 4) AS avg_overall_satisfaction,
  ROUND(
    AVG(
      (
        overall_satisfaction_rating
          + work_quality_rating
          + cleanliness_rating
          + communication_rating
          + delivered_on_time_rating
      )
        / 5.0
    ),
    4
  ) AS avg_survey_score,
  RANK() OVER (ORDER BY AVG(overall_satisfaction_rating) DESC) AS store_survey_score_rank
FROM
  workspace.gold.data_cube
WHERE
  responded_flag is TRUE
GROUP BY
  store_id;